<a href="https://colab.research.google.com/github/Text-Machine/mask-predict/blob/main/example_analysis_colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/> </a>

Uncomment cells bellow when working in Colab

In [ ]:
!gdown 1UvpDyAf_mNSIXBGUnfK3NdKvYTQ4KB7O

In [ ]:
!gdown 1vWP26wx1DlGjK-4JHzvguanA9SPRVnnO

In [ ]:
!git clone https://github.com/Text-Machine/mask-predict.git

In [ ]:
%cd mask-predict

In [ ]:
!pip install -e .

In [1]:
import pandas as pd
import json
from tqdm import tqdm
from explain import *
from pathlib import Path

In [40]:
collection,genre_suffix = 'blb',''
if collection == 'blb':
  genre_suffix = '_with_genre'

maskedToken = 'machine'
modelName = "Livingwithmachines/bert_1760_1900"
dataPath = 'masking_data' # change to '.' when working in colab 
processedFolder = 'processed_data' # change '.' when working in colab
predCol = "pred_bert_1760_1900"

In [ ]:
!unzip -o "{collection}_{maskedToken}_clusters{genre_suffix}.tsv.zip"

In [ ]:
!unzip -o "results_{collection}_{maskedToken}_constrastive.csv.zip"

In [8]:
# load the original sentences with the predicted tokens
df_sentences = pd.read_csv(f'{dataPath}/{collection}_{maskedToken}_clusters{genre_suffix}.tsv', index_col=0, sep='\t')
df_sentences.shape

(131002, 33)

In [50]:
# load the data with the constrastive explanations
results_df = pd.read_csv(f'{processedFolder}/results_{collection}_{maskedToken}_constrastive.csv', index_col=0 )
results_df.shape

(11008832, 4)

In [ ]:
# inspect the data
# each sentence appears twice, once for the original masked token, (Target=maskedtoken) and once for the contrastive token (Target=contrastive)
# i.e. if masked token is 'machine' then contrastive token is 'slave'
results_df.head()

,Token,Score,Target,id
0,the,0.055053,machine,60000
1,[MASK],0.000000,machine,60000
2,shops,0.330447,machine,60000
3,occupy,-0.067776,machine,60000
4,three,0.071201,machine,60000


In [9]:
# inspect alignment of sentences between dataframes df_sentences and results_df
idx = df_sentences.shape[0] -2
list(results_df[results_df.id == idx].Token)[:5],df_sentences.iloc[idx].maskedSentence

(['you', 'would', 'fain', 'be', 'thought'],
 'You would fain be thought to take no share in government , while in reality you are tho main spring of the [MASK] .')

In [11]:
# there is a tiny bug, I forgot the process the last sentence of df_sentences
results_df['id'].nunique(),len(df_sentences)

(131001, 131002)

# Macro-Scale Influences

At the level of all sentences containing the masked token, which words influence the prediction of observed (or counter-factual) terms.

Why is this interesting?
- **validation**: partly shows that the method works, e.g. the words that influence the prediction of the actual masked token word 'machine' should make sense.
- **counterfactual**: highlight which words in the context of x ('machine') are associated with contrastive term y ('slave'), e.g. which words in the context of machine do contribute to the prediction of slavery, foreground intermingling of two discourses?


In [12]:
# divide results_df into two dataframes, one for the masked token and one for the contrastive token, and calculate the average score for each token in each dataframe
slave_df = results_df[(results_df['Target'] == 'slave') | (results_df['Target'] == 'slaves')].groupby('Token').agg(avg=('Score', 'mean'), count=('Score', 'size'), std=('Score', 'std'))
machine_df = results_df[(results_df['Target'] == 'machine') | (results_df['Target'] == 'machines')].groupby('Token').agg(avg=('Score', 'mean'), count=('Score', 'size'), std=('Score', 'std'))

In [13]:
machine_df.shape, slave_df.shape

((74140, 3), (74140, 3))

In [ ]:
min_freq = 10 # how often should the token appear in the explanations to be included in the analysis
top_n = 10 # how many tokens to show in the top list
# show the top contributors to the prediction of the masked token 'slave' (or 'slaves')
slave_df[slave_df['count'] > min_freq
                ].sort_values('avg', ascending=False
                              ).head(top_n)[['avg', 'count', 'std']]

,avg,count,std
Token,,,
oratorial,0.358861,18,0.258491
tentering,0.356591,15,0.205735
folard,0.355547,20,0.207538
exploders,0.346915,14,0.245941
wreath,0.340674,14,0.373802
vanes,0.312471,23,0.223666
werder,0.305503,11,0.262240
bureaucratic,0.303477,20,0.118761
infernal,0.291107,1314,0.165619


In [16]:
# show the top contributors to the prediction of the masked token 'slave' (or 'slaves')
machine_df[machine_df['count'] > min_freq
                ].sort_values('avg', ascending=False
                              ).head(top_n)[['avg', 'count', 'std']]

,avg,count,std
Token,,,
sewing,0.471022,3870,0.250007
tentering,0.430624,15,0.181372
bathing,0.417719,2452,0.265310
dredging,0.411902,518,0.176709
calculating,0.383839,278,0.235466
electrifying,0.378359,101,0.150086
dling,0.378285,80,0.191377
bureaucratic,0.376194,20,0.103513
administrative,0.374670,79,0.233919


# Inspect specific token contributions by sentence

Based on these overall contributions, you can select context words and investigate how they interact with slave and machine in concrete sentences.

In [18]:
explainer = MaskedLMExplainer(model_name=modelName, device=pick_device())

Loading weights:   0%|          | 0/204 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie bert.embeddings.word_embeddings.weight to cls.predictions.decoder.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie cls.predictions.bias to cls.predictions.decoder.bias, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
BertForMaskedLM LOAD REPORT from: Livingwithmachines/bert_1760_1900
Key                      | Status     |  | 
-------------------------+------------+--+-
bert.pooler.dense.weight | UNEXPECTED |  | 
bert.pooler.dense.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [19]:
# get the ids for the sentences containing the context token 'tentering' and the masked tokens 'slave' or 'slaves'
context_token = 'tentering' 
masked_tokens = ['slave','slaves']

results_df[(results_df.Token==context_token) & (results_df.Target.isin(masked_tokens))]['id'].unique()

array([ 48682, 107285, 112473,  15621,  15622,  15627,  15628,  15634,
        15635,  15641,  16153,  16154,  16257,  16261,  16266])

In [22]:
# select one of the above ids and highlight the context tokens in the sentence
idx = 48682
sentence = ' '.join(results_df[(results_df.id==idx) & results_df.Target.isin(['slave','slaves'])]['Token'].values)
target = results_df[(results_df.id==idx) & results_df.Target.isin(['slave','slaves'])]['Target'].values[0]
highlight_context_tokens(explainer, sentence, target=target, word_agg="max")

Explaining:   0%|          | 0/1 [00:00<?, ?it/s]

'\n    <div id="tokviz_88390c6819d240f8bcfcd85c5fa246f3">\n      <div style=\'margin-bottom:6px;\'>\n        <b>Target:</b> <code>slaves</code>\n      </div>\n      <div style=\'margin:6px 0 10px 0; font-size:13px; display:flex; gap:10px; align-items:center;\'>\n        <span style=\'background:rgba(30,136,229,0.35); padding:2px 8px; border-radius:4px;\'>&#9646; predicts</span>\n        <span style=\'background:rgba(229,57,53,0.35);  padding:2px 8px; border-radius:4px;\'>&#9646; opposes</span>\n        <span style=\'background:rgba(255,193,7,0.85);  padding:2px 8px; border-radius:4px; font-weight:bold;\'>[target] mask position</span>\n      </div>\n      <div style=\'line-height:2.4; font-size:15px;\'>\n        <span class=\'tok\' data-score=\'-0.104228\' style=\'background:rgba(229, 57, 53, 0.293); padding:2px 4px; margin:1px; border-radius:4px; cursor:default;\'>the</span> <span class=\'tok\' data-score=\'-0.088744\' style=\'background:rgba(229, 57, 53, 0.264); padding:2px 4px; margi

# Macro-level differences between actual and counterfactual masked tokens


Previous analysis shows contributions for each masked tokens (actual and counterfactual). However, some words might contribute equally to both masked target words.

Therefore we now focus on differences (the contrast): i.e. which words are most associated with either 'slave' or 'machine'.

In [23]:
slave_tok_df = results_df[(results_df['Target'].isin(['slave', 'slaves']))].reset_index(drop=True)
machine_tok_df = results_df[(results_df['Target'].isin(['machine', 'machines']))].reset_index(drop=True)

In [24]:
slave_tok_df['Diff_Score'] = slave_tok_df['Score'] - machine_tok_df['Score']
slave_tok_df['Machine_Score'] = machine_tok_df['Score']
slave_diff = slave_tok_df.groupby('Token').agg(avg_diff=('Diff_Score', 'mean'), count=('Diff_Score', 'size'), std_diff=('Diff_Score', 'std'), slave_scores=('Score', 'mean'), machine_scores=('Machine_Score', 'mean'))#.sort_values('avg_diff', ascending=False).head(10)

In [25]:
slave_tok_df.columns

Index(['Token', 'Score', 'Target', 'id', 'Diff_Score', 'Machine_Score'], dtype='str')

In [26]:
slave_diff[slave_diff['count'] > 10].sort_values('avg_diff', ascending=False).head(5)

,avg_diff,count,std_diff,slave_scores,machine_scores
Token,,,,,
arabian,0.259229,16,0.183309,0.255436,-0.003792
superiors,0.221889,16,0.215575,0.178537,-0.043352
familiarity,0.199497,11,0.252397,0.209070,0.009573
tucker,0.184309,22,0.319949,0.111735,-0.072574
roughing,0.180935,19,0.176782,0.288103,0.107167


# Sentence-level differences

For which sentences do we observe the largest shift in context contribution when switching between different target words?

Shifts are often largest for shorter sentences. Therefore, I added threshold to include only sentence above a certain tokens length.

In [38]:
diff_dict = {'slave':'machine', 'slaves':'machines','machine':'slave', 'machines':'slaves'}
masked_token_dict = {'machine':['machine', 'machines'], 'slave':['slave', 'slaves']}

In [33]:
min_length = 20 # threshold to filter out sentences that are too short 
top_n = 10 # show the top n sentences with the highest average difference in scores between the masked token and the contrastive token
by_sent = slave_tok_df.groupby('id').agg(avg_diff=('Diff_Score', 'mean'), count=('Diff_Score', 'size'))#
by_sent[by_sent['count'] > min_length].sort_values('avg_diff', ascending=False).head(top_n)

,avg_diff,count
id,,
117986,0.200135,23
32600,0.173678,34
32602,0.173678,34
32598,0.173678,34
35872,0.140207,21
35882,0.140207,21
60896,0.137750,27
31708,0.137745,22
31711,0.137745,22


In [42]:
idx = 117986 # select one of the adove ids with a high average difference in scores between the masked token and the contrastive token
sentence = ' '.join(results_df[(results_df.id==idx) & results_df.Target.isin(masked_token_dict[maskedToken])]['Token'].values)
target = results_df[(results_df.id==idx) & results_df.Target.isin(masked_token_dict[maskedToken])]['Target'].values[0]
#print(sentence, target)
highlight_context_tokens(explainer, sentence, target=target, word_agg="max")

Explaining:   0%|          | 0/1 [00:00<?, ?it/s]

'\n    <div id="tokviz_097634508f9a48a89562b6e254f20719">\n      <div style=\'margin-bottom:6px;\'>\n        <b>Target:</b> <code>machine</code>\n      </div>\n      <div style=\'margin:6px 0 10px 0; font-size:13px; display:flex; gap:10px; align-items:center;\'>\n        <span style=\'background:rgba(30,136,229,0.35); padding:2px 8px; border-radius:4px;\'>&#9646; predicts</span>\n        <span style=\'background:rgba(229,57,53,0.35);  padding:2px 8px; border-radius:4px;\'>&#9646; opposes</span>\n        <span style=\'background:rgba(255,193,7,0.85);  padding:2px 8px; border-radius:4px; font-weight:bold;\'>[target] mask position</span>\n      </div>\n      <div style=\'line-height:2.4; font-size:15px;\'>\n        <span class=\'tok\' data-score=\'-0.000676\' style=\'background:rgba(229, 57, 53, 0.101); padding:2px 4px; margin:1px; border-radius:4px; cursor:default;\'>a</span> <span class=\'tok\' data-score=\'0.093441\' style=\'background:rgba(30, 136, 229, 0.192); padding:2px 4px; margin

In [43]:
highlight_context_tokens(explainer, sentence, target=diff_dict[target], word_agg="max")

Explaining:   0%|          | 0/1 [00:00<?, ?it/s]

'\n    <div id="tokviz_a7650a5e19044f208b8d66c3e80d9d20">\n      <div style=\'margin-bottom:6px;\'>\n        <b>Target:</b> <code>slave</code>\n      </div>\n      <div style=\'margin:6px 0 10px 0; font-size:13px; display:flex; gap:10px; align-items:center;\'>\n        <span style=\'background:rgba(30,136,229,0.35); padding:2px 8px; border-radius:4px;\'>&#9646; predicts</span>\n        <span style=\'background:rgba(229,57,53,0.35);  padding:2px 8px; border-radius:4px;\'>&#9646; opposes</span>\n        <span style=\'background:rgba(255,193,7,0.85);  padding:2px 8px; border-radius:4px; font-weight:bold;\'>[target] mask position</span>\n      </div>\n      <div style=\'line-height:2.4; font-size:15px;\'>\n        <span class=\'tok\' data-score=\'0.312877\' style=\'background:rgba(30, 136, 229, 0.586); padding:2px 4px; margin:1px; border-radius:4px; cursor:default;\'>a</span> <span class=\'tok\' data-score=\'0.482439\' style=\'background:rgba(30, 136, 229, 0.850); padding:2px 4px; margin:1

# Overlapping contributors

Which words are strong predictors of the actual and counterfactual token?

Here we look at the top_n contributes for both 'machine' and 'slave' (which appear more than n_count times in the context of the maskedToken) and take the overlap between the top ranked predictors.

In [46]:
min_count = 25 # how often should token appear in the explanations to be included in the analysis of the top contributing tokens to the prediction of the masked token
top_n = 100 # look at top n contr
top_machine_indices = machine_df[machine_df['count'] > min_count].sort_values('avg', ascending=False).head(top_n).index
top_slave_indices = slave_df[slave_df['count'] > min_count].sort_values('avg', ascending=False).head(top_n).index
overlap_indices = top_machine_indices.intersection(top_slave_indices)

In [47]:
overlap_indices

Index(['electrifying', 'dling', 'slaying', 'threshing', 'soulless', 'guns',
       'pendulum', 'electrical', 'jigging', 'cone', 'photographic', 'gun',
       'political', 'copying', 'mowing', 'plug', 'camera', 'glove', 'ry',
       'infernal', 'organism', 'blackwood', 'crane', 'kneading', 'winnowing',
       'praying', 'reaping', 'riddle', 'unoiled', 'pan', 'machines', 'drill',
       'mere', 'trical', 'stump', 'trough', 'feat', 'compartment', 'fans',
       'vanners', 'machine', 'une', 'frame'],
      dtype='str', name='Token')

# Using Metadata and Cluster Scores

## Time

Here we focus on sentence published between start_year and end_year

In [51]:
start_year = 1800
end_year = 1810
selected_sentences = df_sentences[df_sentences['date'].between(start_year, end_year)].index

In [52]:
slave_df = results_df[(results_df['Target'].isin(['slave', 'slaves'])) & (results_df['id'].isin(selected_sentences))].groupby('Token').agg(avg=('Score', 'mean'), count=('Score', 'size'), std=('Score', 'std'))
machine_df = results_df[(results_df['Target'].isin(['machine', 'machines'])) & (results_df['id'].isin(selected_sentences))].groupby('Token').agg(avg=('Score', 'mean'), count=('Score', 'size'), std=('Score', 'std'))

In [53]:
min_count = 10 # how often should token appear in the explanations to be included in the analysis of the top contributing
top_n = 10
slave_df[slave_df['count'] > min_count
                ].sort_values('avg', ascending=False
                              ).head(top_n)[['avg', 'count', 'std']]

,avg,count,std
Token,,,
hydraulic,0.362136,38,0.206643
infernal,0.311885,19,0.189531
electrical,0.289307,17,0.176692
pump,0.279499,14,0.225323
threshing,0.277673,19,0.240998
ry,0.263159,11,0.286471
pan,0.259594,11,0.231331
mere,0.234311,59,0.218953
clocks,0.219860,12,0.256600


In [54]:
machine_df[machine_df['count'] > min_count
                ].sort_values('avg', ascending=False
                              ).head(top_n)[['avg', 'count', 'std']]

,avg,count,std
Token,,,
electrical,0.386584,17,0.180264
dredging,0.384913,22,0.152761
shaft,0.376188,12,0.308198
hydraulic,0.342299,38,0.221731
pump,0.331008,14,0.281246
political,0.313466,46,0.217951
pan,0.307325,11,0.234481
engines,0.305669,33,0.214215
threshing,0.299179,19,0.176211


## Cluster-based filtering

Let's focus only on the sentence that scored high for the 'slave' cluster in the df_sentences dataframe.

In [55]:
df_sentences.columns

Index(['article_path', 'identifier', 'date', 'shelfmarks', 'publisher',
       'title', 'edition', 'contributors', 'creators', 'prevSentence',
       'currentSentence', 'markedSentence', 'maskedSentence', 'nextSentence',
       'targetExpression', 'sentId', 'synt', 'query_label',
       'pred_bert_1760_1900', 'pred_bert_contemporary', 'machine_1760_1900',
       'boy_1760_1900', 'girl_1760_1900', 'slave_1760_1900',
       'artisan_1760_1900', 'woman_1760_1900', 'machine_contemporary',
       'boy_contemporary', 'girl_contemporary', 'slave_contemporary',
       'artisan_contemporary', 'woman_contemporary', 'genre'],
      dtype='str')

In [ ]:
top_cl_sent = 1000
selected_sentences = df_sentences.sort_values(by='slave_1760_1900', ascending=False)[:top_cl_sent].index

In [59]:
slave_df = results_df[(results_df['Target'].isin(['slave', 'slaves'])) & (results_df['id'].isin(selected_sentences))].groupby('Token').agg(avg=('Score', 'mean'), count=('Score', 'size'), std=('Score', 'std'))
machine_df = results_df[(results_df['Target'].isin(['machine', 'machines'])) & (results_df['id'].isin(selected_sentences))].groupby('Token').agg(avg=('Score', 'mean'), count=('Score', 'size'), std=('Score', 'std'))

In [60]:
min_count = 10 # how often should token appear in the explanations to be included in the analysis of the top contributing
top_n = 10
slave_df[slave_df['count'] > min_count
                ].sort_values('avg', ascending=False
                              ).head(top_n)[['avg', 'count', 'std']]

,avg,count,std
Token,,,
mere,0.392088,39,0.178558
slaves,0.317902,14,0.173505
slave,0.286333,26,0.203814
human,0.177345,11,0.114252
they,0.131860,41,0.091019
as,0.131173,43,0.110920
will,0.115934,12,0.127325
a,0.114550,99,0.148179
made,0.106736,17,0.065042


In [61]:
machine_df[machine_df['count'] > min_count
                ].sort_values('avg', ascending=False
                              ).head(top_n)[['avg', 'count', 'std']]

,avg,count,std
Token,,,
mere,0.414041,39,0.173820
slaves,0.389426,14,0.205203
slave,0.269226,26,0.209313
they,0.154508,41,0.121665
made,0.127566,17,0.076379
like,0.121695,11,0.104884
them,0.118649,13,0.127847
will,0.106414,12,0.106104
a,0.103014,99,0.105159


# Fin.